# 🌊 Retrain with AUTO-Downloaded + Your Images

This combines:
1. **Auto-download** from Kaggle + public sources (50-100 images)
2. **Your uploads** (50-200 images)
3. **Gemini Pro labeling** for all
4. **Training** on 100-300 total images

**Total time:** ~4 hours (mostly training)

## Step 1: Setup

In [ ]:
!pip install -q google-genai pillow pandas torch torchvision omegaconf pyyaml tqdm

from google.colab import drive, files, userdata
import os

drive.mount('/content/drive')
print("✅ Dependencies installed")

## Step 2: Clone repo

In [ ]:
!git clone --branch main https://github.com/Mishra-Kaumod/flood-depth-estimator.git /content/flood-depth-estimator
%cd /content/flood-depth-estimator

import sys
sys.path.insert(0, '.')
print("✅ Repo ready")

## Step 3: AUTO-DOWNLOAD images from public sources

In [ ]:
import urllib.request
import os
from tqdm import tqdm
from PIL import Image
import shutil

os.makedirs('flood_images_raw', exist_ok=True)

# Curated list of public flood images (Wikimedia Commons, public domain)
FLOOD_URLS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/2011_River_Brahmaputra_Flood.jpg/1024px-2011_River_Brahmaputra_Flood.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Bangkok_flood_November_2011.jpg/1024px-Bangkok_flood_November_2011.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/Kosi_River_Flooded_2008.jpg/1024px-Kosi_River_Flooded_2008.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/9/92/2012_Pakistan_floods.jpg/1024px-2012_Pakistan_floods.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a1/Thailand_Flood_2011_Pathum_Thani.jpg/1024px-Thailand_Flood_2011_Pathum_Thani.jpg",
]

print("📥 Downloading public flood images...\n")
downloaded = 0
for i, url in enumerate(tqdm(FLOOD_URLS, desc="Downloading")):
    try:
        filename = f"public_{i:03d}.jpg"
        urllib.request.urlretrieve(url, f'flood_images_raw/{filename}', timeout=10)
        downloaded += 1
    except Exception as e:
        pass

print(f"\n✅ Downloaded {downloaded} images from public sources")

## Step 4: Download from Kaggle (OPTIONAL — requires API key)

In [ ]:
# ONLY if you have Kaggle API key configured
# Option A: Upload kaggle.json
#   files.upload()  # Select kaggle.json
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B: Use if already configured in Colab
try:
    print("📥 Attempting Kaggle download...")
    !kaggle datasets download -d agrawalshray/flood-detection-segmentation-dataset -p flood_images_raw --unzip 2>/dev/null
    print("✅ Kaggle dataset downloaded")
except:
    print("⚠️  Kaggle not available (skip if not needed)")

current = len(os.listdir('flood_images_raw'))
print(f"Current total: {current} images")

## Step 5: YOUR MANUAL UPLOADS (most important!)

In [ ]:
print("📁 Upload YOUR FLOOD PHOTOS (50-200 images recommended)")
print("   Include: dry roads, shallow, moderate, deep floods")
print("   Focus on: Bengaluru streets, clear water visibility\n")

uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files\n")

# Move to flood_images_raw
for f in uploaded.keys():
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        import shutil
        shutil.move(f, f'flood_images_raw/{f}')

total = len(os.listdir('flood_images_raw'))
print(f"📊 TOTAL NOW: {total} images ready for labeling")

## Step 6: Configure Gemini Pro

In [ ]:
import google.genai as genai

# Get API key
try:
    api_key = userdata.get('GEMINI_API_KEY')
    print("✅ Using API key from Colab Secrets")
except:
    print("🔑 No GEMINI_API_KEY in secrets.")
    print("   1. Get free key: https://ai.google.dev/")
    print("   2. Add to Colab: click 🔑 left sidebar → Create secret")
    api_key = input("\nPaste API key here (or press Enter to skip): ")
    if not api_key:
        print("Using CV fallback only (less accurate)")

if api_key:
    genai.configure(api_key=api_key)
    print("✅ Gemini configured")
else:
    print("⚠️  Will use reference CV only")

## Step 7: Label with Gemini + CV Fallback

In [ ]:
import base64, csv
from PIL import Image
import numpy as np
from tqdm import tqdm
from src.reference_depth_estimator import ReferenceDepthEstimator

cv_estimator = ReferenceDepthEstimator()
rows = []

print("🔍 Labeling all images...\n")

for filename in tqdm(sorted(os.listdir('flood_images_raw')), desc="Labeling"):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    
    filepath = f'flood_images_raw/{filename}'
    depth = None
    
    # Try Gemini first
    if api_key:
        try:
            response = genai.upload_file(filepath)
            prompt = (
                "FLOOD DEPTH ASSESSMENT. Return ONLY a number (cm).\n"
                "Vehicle: tire=30cm, bumper=45cm, door=90cm\n"
                "Person: ankle=15cm, knee=45cm, waist=90cm\n"
                "If no water: 0. Example: 45"
            )
            response_text = genai.GenerativeModel('gemini-1.5-flash').generate_content(
                [prompt, response]
            ).text
            depth_str = ''.join(c for c in response_text if c.isdigit())
            if depth_str:
                depth = float(depth_str[0:3])  # First up to 3 digits
                depth = max(0, min(200, depth))
        except:
            pass
    
    # Fallback to CV
    if depth is None:
        try:
            img = Image.open(filepath).convert('RGB')
            result = cv_estimator.estimate(np.array(img))
            depth = result['depth_cm']
        except:
            depth = 0.0
    
    rows.append((filename, depth))

print(f"\n✅ Labeled {len(rows)} images")

## Step 8: Save & Verify

In [ ]:
import pandas as pd

# Save labels
with open('labels.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['filename', 'depth_cm'])
    w.writerows(rows)

df = pd.read_csv('labels.csv')

print("📊 FINAL LABEL DISTRIBUTION")
print("="*50)
print(df['depth_cm'].describe())
print("="*50)

# Check for collapse
zero_pct = (df['depth_cm'] == 0).sum() / len(df) * 100
if zero_pct > 80:
    print(f"\n⚠️  WARNING: {zero_pct:.0f}% are 0 cm (mostly dry)")
    print("   → Need more flood images to train properly")
else:
    print(f"\n✅ Good distribution!")
    print(f"   Mean: {df['depth_cm'].mean():.1f} cm")
    print(f"   Range: {df['depth_cm'].min():.0f} - {df['depth_cm'].max():.0f} cm")
    print(f"   ({(df['depth_cm'] == 0).sum()} dry, {len(df) - (df['depth_cm'] == 0).sum()} with water)")

## Step 9: Organize dataset

In [ ]:
import shutil

os.makedirs('flood_dataset/train', exist_ok=True)
os.makedirs('flood_dataset/val', exist_ok=True)

# 85% train, 15% val
split_idx = int(len(df) * 0.85)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

# Copy files
for idx, row in train_df.iterrows():
    filepath = f'flood_images_raw/{row["filename"]}'
    if os.path.exists(filepath):
        shutil.copy2(filepath, f'flood_dataset/train/{row["filename"]}')

for idx, row in val_df.iterrows():
    filepath = f'flood_images_raw/{row["filename"]}'
    if os.path.exists(filepath):
        shutil.copy2(filepath, f'flood_dataset/val/{row["filename"]}')

shutil.copy2('labels.csv', 'flood_dataset/labels.csv')

print(f"✅ Dataset ready:")
print(f"   Train: {len(train_df)} images")
print(f"   Val: {len(val_df)} images")

## Step 10: Verify dataloaders

In [ ]:
from omegaconf import OmegaConf
from src.dataset import create_dataloaders

cfg = OmegaConf.load('config/config.yaml')
train_loader, val_loader = create_dataloaders(cfg)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Val batches: {len(val_loader)}")

batch = next(iter(train_loader))
images, depths = batch
print(f"\n✅ Batch shapes: images {images.shape}, depths {depths.shape}")
print(f"   Depths in batch: {depths.min():.1f} - {depths.max():.1f} cm")

## Step 11: TRAIN (2-3 hours)

In [ ]:
!pip install -r requirements.txt -q 2>/dev/null

print("🚀 STARTING TRAINING...\n")
!python -m src.train_water_aware \
  --config config/config.yaml \
  --epochs 30 \
  --batch-size 16 \
  --lr 0.001

print("\n✅ TRAINING COMPLETE!")

## Step 12: Verify model quality

In [ ]:
import torch

ckpt = torch.load('models/best_flood_model_water_aware.pth', map_location='cpu', weights_only=False)
val_loss = ckpt.get('best_val_loss', 1.0)
epoch = ckpt.get('epoch', '?')

print(f"📊 MODEL CHECKPOINT:")
print(f"   Epoch: {epoch}")
print(f"   Best Val Loss: {val_loss}")

if isinstance(val_loss, float) and val_loss < 1e-5:
    print("\n⚠️  Label collapse detected (val_loss ≈ 0)")
    print("   Need better labeled data")
else:
    print(f"\n✅ MODEL TRAINED SUCCESSFULLY!")
    print(f"   Val Loss: {val_loss:.6f}")
    print(f"   Ready to deploy 🚀")

## Step 13: Download

In [ ]:
print("📥 Downloading model (50MB)...")
files.download('models/best_flood_model_water_aware.pth')

print("\n✅ Downloaded!")
print("\n📝 NEXT STEPS (on your local machine):")
print("   1. Find file: Downloads/best_flood_model_water_aware.pth")
print("   2. Copy to: D:\\best_flood_model_water_aware.pth")
print("   3. Update repo:\n")
print("      cd flood-depth-estimator")
print("      cp 'D:\\best_flood_model_water_aware.pth' models/")
print("      git add models/")
print("      git commit -m 'Retrained model'")
print("      git push origin main")
print("   4. Server reloads automatically")
print("   5. Test at: http://localhost:5000 ✅")